# Tarea · Clase 5 · Un modelo de punta a punta

**Machine Learning** · Programa de Especialización Data Scientist · Docente: **Josef Renato Rodriguez**

**Alumno:** _escribe aquí tu nombre_

---

## El encargo

Una casa de empeño quiere tasar diamantes automáticamente antes de aceptarlos como garantía. Te
entregan los precios de casi 54 000 diamantes de talla redonda, con su peso, su calidad de talla, su
color, su pureza y sus medidas. Tu trabajo es construir el modelo **siguiendo los 8 pasos de la
Clase 5** y reportar qué tan bien tasa.

## Reglas

- Usa `SEED = 42` en todo lo que tenga azar.
- La prueba es el **20 %** de los datos, y se separa en el Paso 3, antes de explorar.
- La prueba no se toca hasta el Paso 8, y ahí se usa **una sola vez**.
- Todo lo que se aprende de los datos (escalas, codificaciones, rellenos) va **dentro de un pipeline**.
- Cada paso termina con una celda de texto con **tus conclusiones**: qué viste y qué decidiste.

## Requisitos mínimos

- Paso 7: compara con validación cruzada de 5 particiones **la vara** (siempre la mediana) y **al menos
  tres modelos del Módulo 1**, con al menos uno lineal y al menos un ensemble (Random Forest o XGBoost).
- Paso 8: reporta MAE, RMSE, R² y MAPE en prueba, el error por quintil de precio y una conclusión
  de negocio en 3 a 5 líneas.

## Pistas

- Hay **al menos tres problemas de calidad** en los datos. Encuéntralos y decide qué hacer con cada uno,
  y por qué. Revisa el diccionario: algunas medidas no pueden ser cero.
- `cut`, `color` y `clarity` son categorías **con orden**. El diccionario dice cuál es.
- Con 43 000 filas, Random Forest tarda: usa `n_estimators=100` y `n_jobs=-1`. SVR no es viable con todas
  las filas; si quieres probarlo, entrénalo con una muestra de 5 000.

## Rúbrica (20 puntos)

| Paso | Qué se evalúa | Puntos |
|---|---|---|
| 1 | Decisión, objetivo, métricas, vara y alcance, escritos antes de abrir los datos | 2 |
| 2 | Tamaño, duplicados, faltantes, forma del objetivo, y los problemas de calidad detectados | 2 |
| 3 | Corte 80/20 hecho antes de explorar, estratificado por quintiles del precio | 2 |
| 4 | Exploración solo con entrenamiento: al menos 2 gráficos y sus conclusiones | 2 |
| 5 | Limpieza justificada, al menos una variable creada y las categorías ordinales bien codificadas | 3 |
| 6 | Pipeline completo, sin fuga de información | 3 |
| 7 | Validación cruzada con la vara y al menos 3 modelos, tabla comparativa y elección justificada | 3 |
| 8 | Prueba usada una vez, las 4 métricas, error por segmento y conclusión de negocio | 3 |
| | **Total** | **20** |

## Entrega

- **Fecha límite:** ______________________
- **Archivo:** este notebook **ejecutado de principio a fin**, con el nombre `Apellido_Nombre_Tarea05.ipynb`.
- Un notebook que no corre de arriba abajo sin errores pierde los puntos de los pasos que no se ejecutan.

---
## Celda 0 · Librerías y datos

Ya está lista. Si no se puede descargar, sube `diamonds.csv` con el botón de archivos de Colab.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             mean_absolute_percentage_error)
import warnings
warnings.filterwarnings("ignore")

SEED = 42

def metricas(y_real, y_pred):
    return {"MAE": mean_absolute_error(y_real, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_real, y_pred)),
            "R2": r2_score(y_real, y_pred),
            "MAPE": mean_absolute_percentage_error(y_real, y_pred)}

URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv"
try:
    diamantes = pd.read_csv(URL)
    print("Datos descargados de internet.")
except Exception as e:
    print(f"No se pudo descargar ({type(e).__name__}). Usando el archivo local.")
    diamantes = pd.read_csv("diamonds.csv")
diamantes.head()

---
# Paso 1 · Definir el problema y la métrica

Responde en texto, antes de mirar los datos: ¿qué decisión apoya el modelo?, ¿qué se predice y para qué unidad?, ¿con qué métricas lo vas a medir y por qué?, ¿contra qué vara?, ¿para qué casos no aplicaría?

**Paso 1 · Tus conclusiones**

_Escribe aquí qué viste y qué decidiste._

---
# Paso 2 · Cargar y revisar los datos

Tamaño, tipos de columna, duplicados, faltantes, rangos de cada variable (`describe()`) y la forma del precio (histograma y asimetría, en escala normal y logarítmica). Anota los problemas de calidad que encuentres.

In [ ]:
# Paso 2 · tu código aquí

**Paso 2 · Tus conclusiones**

_Escribe aquí qué viste y qué decidiste._

---
# Paso 3 · Separar la prueba antes de mirar

Corte 80/20 con `random_state=SEED`, estratificado con `pd.qcut(y, 5, labels=False)`. Muestra el tamaño y la mediana del precio en cada parte.

In [ ]:
# Paso 3 · tu código aquí

**Paso 3 · Tus conclusiones**

_Escribe aquí qué viste y qué decidiste._

---
# Paso 4 · Explorar, solo con entrenamiento

Al menos dos gráficos: por ejemplo, precio contra peso (`carat`) y precio por categoría de `cut`, `color` o `clarity`. Calcula las correlaciones con el precio. ¿Algo te sorprende?

In [ ]:
# Paso 4 · tu código aquí

**Paso 4 · Tus conclusiones**

_Escribe aquí qué viste y qué decidiste._

---
# Paso 5 · Limpiar y crear variables

Decide qué hacer con cada problema de calidad del Paso 2 y aplícalo a entrenamiento y prueba con la misma regla. Escribe una función `preparar(df)` que cree al menos una variable nueva y que no aprenda nada de los datos.

In [ ]:
# Paso 5 · tu código aquí

**Paso 5 · Tus conclusiones**

_Escribe aquí qué viste y qué decidiste._

---
# Paso 6 · Armar el pipeline

`preparar` → `ColumnTransformer` (ordinales con `OrdinalEncoder(categories=...)`, numéricas con escala) → modelo, entrenando con `log(precio)` mediante `TransformedTargetRegressor`. Escribe una función `armar(modelo, log=True)`.

In [ ]:
# Paso 6 · tu código aquí

**Paso 6 · Tus conclusiones**

_Escribe aquí qué viste y qué decidiste._

---
# Paso 7 · Comparar modelos con validación cruzada

`KFold(5, shuffle=True, random_state=SEED)` y `cross_validate` sobre el entrenamiento. La vara y al menos tres modelos (uno lineal y un ensemble como mínimo). Arma una tabla con MAE, su desviación y MAPE. Elige y justifica.

In [ ]:
# Paso 7 · tu código aquí

**Paso 7 · Tus conclusiones**

_Escribe aquí qué viste y qué decidiste._

---
# Paso 8 · Evaluar una vez en prueba y dejarlo listo para usar

Entrena el elegido con todo el entrenamiento, mide en prueba MAE, RMSE, R² y MAPE, compáralo con la vara, calcula el error por quintil de precio y guarda el pipeline con `joblib`. Termina con una conclusión de negocio de 3 a 5 líneas: ¿le darías este modelo a la casa de empeño? ¿Para qué diamantes sí y para cuáles no?

In [ ]:
# Paso 8 · tu código aquí

**Paso 8 · Tus conclusiones**

_Escribe aquí qué viste y qué decidiste._

---
# Para cerrar

Responde en 2 o 3 líneas cada una:

1. ¿Qué paso te costó más, y por qué?
2. ¿Qué cambió más el resultado: el modelo, la limpieza, las variables nuevas o el logaritmo?
3. Si tuvieras que explicarle el modelo a la casa de empeño en una frase, ¿qué le dirías?